In [1]:
# 1) Project setup and imports
from pathlib import Path
import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts" / "train.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
# 2) Import reusable training components from scripts/train.py (with reload)
import sys
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.train as train_mod
importlib.reload(train_mod)

set_seed = train_mod.set_seed
build_dataloaders = train_mod.build_dataloaders
AthenAIModel = train_mod.AthenAIModel
COMMAND_VOCAB = train_mod.COMMAND_VOCAB
TARGET_NUM_SAMPLES = train_mod.TARGET_NUM_SAMPLES
NUM_SENSORS = train_mod.NUM_SENSORS

set_seed(42)
print("Reloaded scripts.train and loaded", len(COMMAND_VOCAB), "command classes")

/home/nithira/Multimodal_Intent_Reconstruction_for_Safety_Critical_Communication_in_Extreme_Acoustic_Environments/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reloaded scripts.train and loaded 20 command classes


In [3]:
# 3) Build dataloaders
BATCH_SIZE = 32
DATA_DIR = PROJECT_ROOT / "data" / "mixed"

dataloaders = build_dataloaders(data_dir=DATA_DIR, batch_size=BATCH_SIZE)
train_loader = dataloaders["train"]
val_loader = dataloaders["val"]
test_loader = dataloaders["test"]

print("Train samples:", len(train_loader.dataset))
print("Val samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

Train samples: 793
Val samples: 93
Test samples: 114


In [4]:
# 4) Build model + optimizer + loss
MODE = "full"  # change to "base" if needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AthenAIModel(mode=MODE).to(device)
trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(trainable_params, lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print("Mode:", MODE)
print("Device:", device)
print("Trainable params:", sum(p.numel() for p in trainable_params))
print("Frozen params:", sum(p.numel() for p in model.parameters() if not p.requires_grad))

Mode: full
Device: cuda
Trainable params: 3461397
Frozen params: 200205824


In [5]:
# 5) One-batch sanity check
audio, sensor, target = next(iter(train_loader))

print("Audio shape:", tuple(audio.shape), "expected [B, 32000]")
print("Sensor shape:", tuple(sensor.shape), "expected [B, 128, 8]")
print("Target shape:", tuple(target.shape))

audio = audio.to(device)
sensor = sensor.to(device)
target = target.to(device)

model.train()
if MODE == "full":
    logits = model(audio, sensor)
else:
    logits = model(audio)

loss = criterion(logits, target)
print("Logits shape:", tuple(logits.shape), "expected [B, 20]")
print("Sanity loss:", float(loss.item()))

Audio shape: (32, 32000) expected [B, 32000]
Sensor shape: (32, 128, 8) expected [B, 128, 8]
Target shape: (32,)


/home/nithira/Multimodal_Intent_Reconstruction_for_Safety_Critical_Communication_in_Extreme_Acoustic_Environments/.venv/lib/python3.10/site-packages/torch/masked/maskedtensor/creation.py:19: UserWarning: The PyTorch API of MaskedTensors is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.masked module for further information about the project.
  return MaskedTensor(data, mask, requires_grad)


Logits shape: (32, 20) expected [B, 20]
Sanity loss: 2.9761033058166504


In [6]:
# 6) Start training from the script (recommended)
import subprocess

cmd = [
    ".venv/bin/python", "scripts/train.py",
    "--mode", MODE,
    "--epochs", "50",
    "--batch_size", str(BATCH_SIZE),
    "--lr", "3e-4",
]

print("Running:", " ".join(cmd))
# Uncomment to launch training from notebook:
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

Running: .venv/bin/python scripts/train.py --mode full --epochs 50 --batch_size 32 --lr 3e-4


Epoch 001 | train_loss=2.8892 | train_acc=0.1009 | val_loss=2.7898 | val_acc=0.0753 | best=updated


Epoch 002 | train_loss=2.8336 | train_acc=0.1110 | val_loss=2.7624 | val_acc=0.1505 | best=updated


Epoch 003 | train_loss=2.8177 | train_acc=0.1148 | val_loss=2.7550 | val_acc=0.1398 | best=updated


Epoch 004 | train_loss=2.8144 | train_acc=0.1375 | val_loss=2.7537 | val_acc=0.1398 | best=updated


...

Epoch 046 | train_loss=1.1612 | train_acc=0.6267 | val_loss=2.9717 | val_acc=0.2366 | patience=9/10


/home/nithira/Multimodal_Intent_Reconstruction_for_Safety_Critical_Communication_in_Extreme_Acoustic_Environments/scripts/train.py:262: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental fe

Epoch 047 | train_loss=1.0938 | train_acc=0.6608 | val_loss=3.0299 | val_acc=0.2151 | patience=10/10
Early stopping triggered at epoch 47; best epoch was 37


Best checkpoint: /home/nithira/Multimodal_Intent_Reconstruction_for_Safety_Critical_Communication_in_Extreme_Acoustic_Environments/checkpoints/best_model.pt
Test loss: 2.7683
Test accuracy: 0.2105

Per-class report:
                      precision    recall  f1-score   support

    stop the machine       0.00      0.00      0.00         5
  emergency shutdown       0.33      0.43      0.38         7
evacuate immediately       0.00      0.00      0.00         3
        reduce speed       0.00      0.00      0.00         5
   increase pressure       0.13      0.40      0.20        10
   decrease pressure       0.00      0.00      0.00         4
          open valve       0.33      0.12      0.18         8
         close valve       1.00      0.11      0.20         9
activate safety lock       0.12      0.25      0.16         8
 release safety lock       0.00      0.00      0.00         4
     call supervisor       0.00      0.00      0.00         6
        check sensor       0.00      0.

CompletedProcess(args=['.venv/bin/python', 'scripts/train.py', '--mode', 'full', '--epochs', '50', '--batch_size', '32', '--lr', '3e-4'], returncode=0)

In [8]:
# 7) Run evaluation from notebook (after training)
eval_cmd = [
    ".venv/bin/python", "scripts/evaluate.py",
    "--mode", MODE,
]

print("Running:", " ".join(eval_cmd))
# Uncomment to launch evaluation from notebook:
subprocess.run(eval_cmd, cwd=PROJECT_ROOT, check=True)

Running: .venv/bin/python scripts/evaluate.py --mode full


/home/nithira/Multimodal_Intent_Reconstruction_for_Safety_Critical_Communication_in_Extreme_Acoustic_Environments/scripts/evaluate.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental

Overall top-1 accuracy: 0.21929824561403508

Classification report:

                      precision    recall  f1-score   support

    stop the machine       0.00      0.00      0.00         5
  emergency shutdown       0.38      0.43      0.40         7
evacuate immediately       0.00      0.00      0.00         3
        reduce speed       0.00      0.00      0.00         5
   increase pressure       0.12      0.40      0.19        10
   decrease pressure       0.00      0.00      0.00         4
          open valve       0.50      0.12      0.20         8
         close valve       1.00      0.11      0.20         9
activate safety lock       0.18      0.38      0.24         8
 release safety lock       0.00      0.00      0.00         4
     call supervisor       0.00      0.00      0.00         6
        check sensor       0.00      0.00      0.00         6
      restart system       0.30      0.43      0.35         7
       halt conveyor       0.22      0.50      0.31         4


CompletedProcess(args=['.venv/bin/python', 'scripts/evaluate.py', '--mode', 'full'], returncode=0)

In [ ]:
# 8) Quick usage guide
print("Run cells 1-5 for setup and sanity checks.")
print("In cell 6, uncomment subprocess.run(...) to start training.")
print("After training, in cell 7 uncomment subprocess.run(...) to run evaluation.")
print("Evaluation outputs: eval_results.json, eval_reliability.png, eval_uncertainty.png")

Run cells 1-5 for setup and sanity checks.
In cell 6, uncomment subprocess.run(...) to start training.
After training, run: .venv/bin/python scripts/evaluate.py --mode full
